## 爬虫实例 1：岭南学院教师名录整理

> *从一个公开页面，提取结构化名录数据，输出 Excel 与 Markdown 两种格式。*

本节提供两种学习路径，内容完全相同，只是呈现方式不同，读者按自身情况选择：

- **路径 A（提示词版）**：适合已完成第一章练习、能独立分析网页结构的读者。直接给出网页观察记录、提示词和代码，节奏紧凑。
- **路径 B（对话版）**：适合零基础或希望理解「提示词是怎么写出来的」的读者。还原学生与 AI 的完整多轮对话，展示信息如何逐步汇聚成提示词。

两条路径共用相同的代码和验证步骤，选一条读完即可。

### 目标分析：这是一个什么样的页面？

访问 <https://lingnan.sysu.edu.cn/Faculty>，用 DevTools 快速判断（操作方法见第一章 1.4 节）：

- 按 `Ctrl+U`（Mac：`Command+U`）查看源码，搜索教师姓名「毕青苗」，能直接找到对应文本 → 数据写在 HTML 里，无需调用 API
- Network 面板无明显 XHR/Fetch 数据请求 → 无 API 接口
- 切换顶部教研室标签时 URL 不变，页面底部无翻页按钮 → 所有教师在同一页面，只需请求一次

这是静态页面爬取的最简单情形：**一次请求 → 解析 HTML → 输出文件**。

用 DevTools 的 Elements 面板检查可知，每位教师对应一个 `div.infors` 块。以下是两条真实 HTML 样例，注意两者结构**略有不同**：

```html
<!-- 样例 A：有职务字段（刘贯春） -->
<div class="infors">
  <h3>
    <a href="/faculty/LiuGuanchun" target="_blank">刘贯春</a>
    <span>教授</span>
  </h3>
  <p class="one-line">宏观经济教研室</p>
  <p class="one-line">宏观经济教研室主任</p>
  <p class="one-line">liugch9@mail.sysu.edu.cn</p>
  <p class="text-light two-line">
    <b>研究方向:</b>财政与金融、企业投融资决策
  </p>
</div>

<!-- 样例 B：无职务字段（毕青苗） -->
<div class="infors">
  <h3>
    <a href="/faculty/BiQingmiao" target="_blank">毕青苗</a>
    <span>副教授</span>
  </h3>
  <p class="one-line">政治经济学教研室</p>
  <p class="one-line">biqm@mail.sysu.edu.cn</p>
  <p class="text-light two-line">
    <b>研究方向:</b>中国经济增长，营商环境建设
  </p>
</div>
```

对比两段 HTML，字段与位置的对应关系如下：

| 字段 | HTML 位置 | 说明 |
|------|----------|------|
| 姓名 | `h3 > a` 标签文本 | 始终存在 |
| 职称 | `h3 > span` 标签文本 | 始终存在 |
| 个人主页链接 | `h3 > a` 的 `href` 属性 | 需拼接域名前缀 |
| 教研室 | 第 1 个 `p.one-line` 文本 | 始终存在，固定位置 |
| 职务 | 第 2 个 `p.one-line`（若不含 `@`） | 部分教师无此字段 |
| 电子邮箱 | 含 `@` 的 `p.one-line` 文本 | 靠内容特征识别 |
| 研究方向 | `p.two-line` 文本，去掉「研究方向:」前缀 | 始终存在 |


> ⚠️ **本例的解析难点**：`class="one-line"` 同时用于教研室、职务、邮箱三种内容，没有独立的 class 可以区分。
> 
> - 正确识别方式：第一个 `p.one-line` 固定是教研室；之后含 `@` 的是邮箱；不含 `@` 的是职务（若存在）。
> - 这一规则必须在提示词里明确说明，否则 AI 生成的代码遇到有职务字段的教师时会把职务误判为邮箱。


## 路径 A：提示词版

### 网页观察记录表

按照第一章 1.6 节的工作流，写提示词之前先把观察结论填表整理：

| 项目 | 内容 |
|------|------|
| 目标网址 | `https://lingnan.sysu.edu.cn/Faculty` |
| 一条记录对应什么 | 一位教师的基本信息 |
| 目标字段清单 | 姓名、职称、教研室、职务、研究方向、电子邮箱、个人主页链接 |
| 是否有分页 | 否，所有教师在同一页面 |
| 是否有详情页 | 有，但目标字段在列表页已完整呈现，无需进入详情页 |
| 数据来源模式 | HTML（模式 A） |
| 关键定位信息 | `div.infors` 为每位教师的外层块 |
| 字段识别方式 | 含 `@` → 邮箱；含「教研室」→ 教研室；其余 `p.one-line` → 职务 |
| 希望输出格式 | Excel（`.xlsx`）+ Markdown（`.md`） |
| 后续分析目标 | 名录整理，供查阅使用 |

### 提示词

基于上表直接写出完整提示词：

> 请帮我用 Python 写一个爬虫，完成以下任务。本代码仅用于课堂教学演示。
>
> **目标网站**：`https://lingnan.sysu.edu.cn/Faculty`，教师名录全部在一个页面内，无需翻页。
>
> **采集对象**：每一位教师的基本信息，每条记录对应一位教师。
>
> **数据格式**：数据直接写在 HTML 中（模式 A）。以下是两条真实 HTML 样例，注意两者结构不同——刘贯春有职务字段，毕青苗没有：
>
> ```html
> <!-- 样例 A：有职务 -->
> <div class="infors">
>   <h3>
>     <a href="/faculty/LiuGuanchun" target="_blank">刘贯春</a>
>     <span>教授</span>
>   </h3>
>   <p class="one-line">宏观经济教研室</p>
>   <p class="one-line">宏观经济教研室主任</p>
>   <p class="one-line">liugch9@mail.sysu.edu.cn</p>
>   <p class="text-light two-line">
>     <b>研究方向:</b>财政与金融、企业投融资决策
>   </p>
> </div>
>
> <!-- 样例 B：无职务 -->
> <div class="infors">
>   <h3>
>     <a href="/faculty/BiQingmiao" target="_blank">毕青苗</a>
>     <span>副教授</span>
>   </h3>
>   <p class="one-line">政治经济学教研室</p>
>   <p class="one-line">biqm@mail.sysu.edu.cn</p>
>   <p class="text-light two-line">
>     <b>研究方向:</b>中国经济增长，营商环境建设
>   </p>
> </div>
> ```
>
> **字段识别规则**（请严格按此逻辑解析）：
> - `h3 > a` 文本为**姓名**；`href` 属性拼接 `https://lingnan.sysu.edu.cn` 后为**个人主页链接**
> - `h3 > span` 文本为**职称**
> - `class="one-line"` 的 `<p>` 标签按顺序处理：第一个文本为**教研室**；之后若存在不含 `@` 的文本则为**职务**；含 `@` 的文本为**电子邮箱**
> - `class` 包含 `two-line` 的 `<p>` 标签内容去掉「研究方向:」前缀后为**研究方向**
> - 某字段不存在时填入空字符串，不跳过整条记录
>
> **需要提取的字段**（共 7 个，按此顺序）：
> `姓名`、`职称`、`教研室`、`职务`、`研究方向`、`电子邮箱`、`个人主页链接`
>
> **技术要求**：
> - 使用 `requests` + `BeautifulSoup`
> - `User-Agent` 设置为真实 Chrome 浏览器值
> - 请求失败时打印错误信息并终止，不静默失败
> - 将解析单条教师信息的逻辑封装为函数 `parse_faculty(card)`
>
> **输出**：
> - 一份 Excel 文件，命名为 `lingnan_faculty.xlsx`，中文字段名
> - 一份 Markdown 文件，命名为 `lingnan_faculty.md`，每位教师用列表格式呈现，邮箱和链接用尖括号包裹（可点击），格式如下：
>
> ```markdown
> - **刘贯春**｜教授
>   - 教研室：宏观经济教研室
>   - 职务：宏观经济教研室主任
>   - 研究方向：财政与金融、企业投融资决策
>   - 邮箱：<liugch9@mail.sysu.edu.cn>
>   - 主页：<https://lingnan.sysu.edu.cn/faculty/LiuGuanchun>
> ```
>
> **代码要求**：加中文注释；先只打印前 3 条结果，确认字段正确后再保存完整文件。

**提示词要点分析：**

- ✅ **提供了两个结构不同的 HTML 样例**——一个有职务（刘贯春），一个没有（毕青苗），覆盖了字段缺失的边界情况
- ✅ **明确了字段识别规则**，尤其是 `class="one-line"` 复用问题——这是本例最容易出错的地方
- ✅ **指定了输出格式的具体样式**，AI 不需要猜 Markdown 格式
- ✅ **要求先打印 3 条再保存**，对应第一章「先验证、再输出」的实践习惯
- ✅ **封装为函数**，便于后续单独修改解析逻辑

---
## 路径 B：对话版

本路径还原一个零基础学生与 AI 从「只有一个网址」到「生成可运行代码」的完整过程，展示提示词是如何在对话中逐步生长出来的。

**阅读方式**：先通读全部六轮对话（观察者模式），理解每轮在解决什么问题；DevTools 的具体操作集中在对话之后（操作者模式），读完对话再去动手，不需要来回切换。

**开始之前**：大多数 AI 的默认行为是「收到任务 → 立即给出完整答案」。在发出第一条消息之前，先把以下**角色设定提示词**单独发给 AI，收到「明白」后再开始对话：

> 接下来我们要合作完成一个网页爬虫任务。请你扮演一位**爬虫开发引导者**，遵守以下规则：
>
> 1. **不要直接生成代码**，除非我明确说「请生成代码」
> 2. 每次回复只问我**一到两个**最关键的问题，等我回答后再继续
> 3. 引导我依次提供：目标网址、需要的字段、数据用途、HTML 结构（需我去 DevTools 查）、是否有翻页、输出格式——但不要一次全问
> 4. 信息足够后，先把收集到的内容**整理成结构化提示词**给我确认，我确认后再生成代码
> 5. 如果我的信息有歧义或可能导致代码出错，请追问，不要自行假设
>
> 收到后请回复「明白，请告诉我你的爬虫任务」。

### 六轮对话

---

**第 1 轮**

🧑 **学生**

> 我想爬取这个网页的信息：`https://lingnan.sysu.edu.cn/Faculty`，帮我写一个 Python 爬虫。（附上页面截图）

🤖 **AI**

> 好的，我看到这是中山大学岭南学院的教师名录页面。在写代码之前，我需要先了解几件事。
>
> **第一个问题：你想提取哪些信息，打算用这份数据做什么？**
>
> 从截图来看，每位教师的卡片里包含姓名、职称、邮箱、研究方向，部分教师还有职务信息（如「教研室主任」）。你需要全部提取，还是只要其中几个字段？了解用途有助于我决定输出格式。

💡 AI 没有直接写代码，而是先问「要什么」和「做什么用」——对应第一章提示词八要素中的要素②（数据字段）和要素⑤（输出要求）。

---

**第 2 轮**

🧑 **学生**

> 我想要：姓名、职称、邮箱、研究方向，还有个人主页链接。主要是整理一份名录存档，方便查阅。输出成 Excel 就好。

🤖 **AI**

> 明白了。现在我需要了解网页的 **HTML 结构**，才能告诉代码去哪里找数据。
>
> 请做一件事：在 Chrome 中打开这个页面，对着任意一位教师的姓名右键 → 选「检查」，找到包含该教师完整信息的外层 `<div>`，右键 → Copy → Copy outerHTML，把复制的内容发给我。具体操作见本节末尾的「DevTools 操作指引」。

💡 AI 需要 HTML 结构才能写出正确的解析逻辑，必须让你去 DevTools 里取——这正是第一章 1.4 节介绍 DevTools 的实战用途。

---

**第 3 轮**

🧑 **学生**

> 我找到了，是这样的：
>
> ```html
> <div class="infors">
>   <h3>
>     <a href="/faculty/LiuGuanchun" target="_blank">刘贯春</a>
>     <span>教授</span>
>   </h3>
>   <p class="one-line">宏观经济教研室</p>
>   <p class="one-line">宏观经济教研室主任</p>
>   <p class="one-line">liugch9@mail.sysu.edu.cn</p>
>   <p class="text-light two-line">
>     <b>研究方向:</b>财政与金融、企业投融资决策
>   </p>
> </div>
> ```

🤖 **AI**

> 结构很清晰。我能看出：姓名在 `h3 > a`，职称在 `h3 > span`，主页链接是 `href` 属性（需拼接域名），研究方向在 `p.two-line`。
>
> **但有一个问题**：`class="one-line"` 的 `<p>` 标签出现了三次——教研室、职务、邮箱各用了一个，同一个 class 对应不同内容（这是第一章误区⑥提到的情况）。
>
> 更重要的是，**我不确定所有教师的结构都一样**。请再帮我做一件事：在页面上找另一位**没有职务信息**的教师，用同样方式复制一段 HTML 给我，让我对比结构是否一致。

💡 AI 主动要求第二个样例——因为只有一个样例时，无法发现字段可能缺失的情况（对应第一章误区⑥）。

---

**第 4 轮**

🧑 **学生**

> 好的，毕青苗的是这样：
>
> ```html
> <div class="infors">
>   <h3>
>     <a href="/faculty/BiQingmiao" target="_blank">毕青苗</a>
>     <span>副教授</span>
>   </h3>
>   <p class="one-line">政治经济学教研室</p>
>   <p class="one-line">biqm@mail.sysu.edu.cn</p>
>   <p class="text-light two-line">
>     <b>研究方向:</b>中国经济增长，营商环境建设
>   </p>
> </div>
> ```

🤖 **AI**

> 两段 HTML 对比，差异很清楚了：
>
> | 字段 | 刘贯春 | 毕青苗 |
> |------|--------|--------|
> | 教研室 | ✅ 第 1 个 `p.one-line` | ✅ 第 1 个 `p.one-line` |
> | 职务 | ✅ 第 2 个 `p.one-line`（不含 `@`） | ❌ 不存在 |
> | 邮箱 | ✅ 第 3 个 `p.one-line`（含 `@`） | ✅ 第 2 个 `p.one-line`（含 `@`） |
>
> 教研室始终是第一个，可以靠位置固定。职务不一定存在，但识别规则简单：第一个之后若还有不含 `@` 的文本就是职务，含 `@` 的就是邮箱。
>
> 还有两个问题：
> 1. 除了 Excel，是否考虑同时输出一份 Markdown 文件？名录类文档用 Markdown 更便于共享（飞书、Notion、GitHub 均可直接渲染）。
> 2. 页面顶部的教研室分类标签，切换时 URL 有没有变化？确认一下是否需要翻页。

💡 AI 主动建议增加 Markdown 输出（要素⑤），并追问翻页问题（要素①）。不确认翻页，代码可能只抓到部分数据。

---

**第 5 轮**

🧑 **学生**

> Markdown 也要，好主意。翻页：我切换了几个教研室标签，URL 没有变化，滑到底部也没有翻页按钮。

🤖 **AI**

> URL 不变、无翻页按钮，说明所有教师数据在同一个 HTML 页面里，只需请求一次。现在信息已经足够，先整理成提示词给你确认：
>
> [整理后的完整提示词见路径 A 的「提示词」小节，内容完全一致]
>
> 确认无误后说「请生成代码」。

💡 在生成代码前先整理提示词、等待确认——这一步让你看清散落在五轮对话中的信息如何汇聚成标准格式。

---

**第 6 轮**

🧑 **学生**

> 好的，请生成代码。

🤖 **AI**：生成代码（见下方「代码」小节）。

---

**六轮复盘**

| 轮次 | 学生提供了什么 | AI 追问了什么 | 对应提示词要素 |
|------|--------------|-------------|---------------|
| 第 1 轮 | 网址 + 截图 | 要哪些字段？做什么用？ | ②⑤ |
| 第 2 轮 | 字段清单 + 用途 | 去 DevTools 取 HTML | ④ |
| 第 3 轮 | 一段 HTML | 结构可能不一致，要第二段 | ② |
| 第 4 轮 | 第二段 HTML | 建议加 Markdown；追问翻页 | ①⑤ |
| 第 5 轮 | 确认无翻页 | 整理提示词，等待确认 | — |
| 第 6 轮 | 确认提示词 | 生成代码 | — |

**规律**：AI 每次追问，恰好对应第一章 1.7 节「提示词通用模板」里缺失的一个要素。理解这个规律后，下次可以反过来：先自己对照模板把空格填好，再发给 AI，直接进入路径 A 的工作方式。

### DevTools 操作指引{.unnumbered}

本节对应第 2 轮对话中「复制 HTML」的操作步骤。DevTools 的基本用法见第一章 1.4 节，这里只补充一个新动作：**如何复制单个元素的完整 HTML**。

**第一步：打开 DevTools，进入元素选择模式**

- Windows：`Ctrl+Shift+C`
- Mac：`Command+Shift+C`

光标变成带箭头的十字形，页面进入可点击状态。

**第二步：点击目标元素，定位 HTML 位置**

在页面上点击任意一位教师的姓名，Elements 面板会自动高亮对应的 `<a>` 标签。

**第三步：向上选取外层 `<div class="infors">`**

Elements 面板高亮的是最内层的 `<a>` 标签。在面板里点击上方的 `<div class="infors">`，选中包含该教师全部信息的外层块。

**第四步：复制 outerHTML**

在 `<div class="infors">` 上右键 → **Copy → Copy outerHTML**，即可复制完整 HTML，直接粘贴给 AI。

> 💡 **为什么要复制外层 `<div>`，而不是直接复制 `<a>` 标签？** `<a>` 标签只包含姓名和链接，AI 看不到邮箱、研究方向等其他字段。复制包含完整信息的外层块，AI 才能分析字段之间的结构关系，写出正确的解析逻辑。

---
## 代码

### 环境准备{.unnumbered}

本节需要以下 Python 库，如果尚未安装，在终端执行：

```bash
pip install requests beautifulsoup4 pandas openpyxl
```

- `requests`：发送 HTTP 请求，获取网页内容
- `beautifulsoup4`：解析 HTML，提取目标字段
- `pandas`：整理数据，输出 Excel
- `openpyxl`：`pandas` 输出 `.xlsx` 格式的底层依赖

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# ── 请求头：模拟真实浏览器，避免被服务器拒绝 ──
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )
}

BASE_URL = 'https://lingnan.sysu.edu.cn'
LIST_URL = f'{BASE_URL}/Faculty'

In [ ]:
def parse_faculty(card) -> dict:
    """
    解析单个教师信息块（div.infors），返回 7 个字段的字典。
    字段缺失时填入空字符串，不抛出异常。

    class="one-line" 同时用于教研室、职务、邮箱三种内容：
    - 第一个固定是教研室（所有教师都有教研室归属）
    - 之后含 @ 的是邮箱
    - 之后不含 @ 的是职务（部分教师有）
    """
    result = {
        '姓名': '', '职称': '', '教研室': '',
        '职务': '', '研究方向': '', '电子邮箱': '', '个人主页链接': ''
    }

    # 姓名 + 个人主页链接
    h3 = card.find('h3')
    if h3:
        a_tag = h3.find('a')
        if a_tag:
            result['姓名'] = a_tag.get_text(strip=True)
            href = a_tag.get('href', '')
            result['个人主页链接'] = BASE_URL + href if href else ''
        # 职称
        span = h3.find('span')
        if span:
            result['职称'] = span.get_text(strip=True)

    # 逐一处理 class="one-line" 的 <p> 标签
    # 第一个固定是教研室；之后含 @ 的是邮箱；不含 @ 的是职务
    one_lines = card.find_all('p', class_='one-line')
    for i, p in enumerate(one_lines):
        text = p.get_text(strip=True)
        if i == 0:
            result['教研室'] = text
        elif '@' in text:
            result['电子邮箱'] = text
        else:
            result['职务'] = text

    # 研究方向：去掉「研究方向:」前缀
    two_line = card.find('p', class_='two-line')
    if two_line:
        raw = two_line.get_text(strip=True)
        result['研究方向'] = raw.replace('研究方向:', '').strip()

    return result


In [ ]:
# ── Step 1：请求页面 ──
print(f'正在请求：{LIST_URL}')
resp = requests.get(LIST_URL, headers=HEADERS, timeout=15)
if resp.status_code != 200:
    print(f'请求失败，状态码：{resp.status_code}')
else:
    resp.encoding = 'utf-8'

    # ── Step 2：定位所有教师信息块 ──
    soup = BeautifulSoup(resp.text, 'html.parser')
    cards = soup.find_all('div', class_='infors')
    print(f'共找到 {len(cards)} 位教师')

    # ── Step 3：先打印前 3 条，验证字段是否正确 ──
    print('\n【前 3 条预览】')
    for card in cards[:3]:
        info = parse_faculty(card)
        for k, v in info.items():
            print(f'  {k}：{v}')
        print()

In [ ]:
# ── Step 4：确认字段无误后，解析全部教师并输出文件 ──
all_faculty = [parse_faculty(card) for card in cards]

# 输出 Excel
df = pd.DataFrame(all_faculty)
df.to_excel('lingnan_faculty.xlsx', index=False)
print(f'✅ Excel 已保存（共 {len(df)} 条记录）')

# 输出 Markdown（列表格式，邮箱和链接用尖括号包裹）
md_lines = ['# 中山大学岭南学院教师名录\n']
for row in all_faculty:
    md_lines.append(f"- **{row['姓名']}**｜{row['职称']}")
    if row['教研室']:
        md_lines.append(f"  - 教研室：{row['教研室']}")
    if row['职务']:
        md_lines.append(f"  - 职务：{row['职务']}")
    if row['研究方向']:
        md_lines.append(f"  - 研究方向：{row['研究方向']}")
    if row['电子邮箱']:
        md_lines.append(f"  - 邮箱：<{row['电子邮箱']}>")
    if row['个人主页链接']:
        md_lines.append(f"  - 主页：<{row['个人主页链接']}>")
    md_lines.append('')

with open('lingnan_faculty.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(md_lines))
print('✅ Markdown 已保存')

---
## 验证与调试

### 爬完先检查，再保存{.unnumbered}

第一章强调：**「运行成功」≠「任务完成」**。代码能跑不代表数据是对的。运行完成后先执行以下检查，确认无误再使用数据：

In [ ]:
print('【1. 数据量】')
print(f'  共 {len(df)} 条记录')

print('\n【2. 字段完整性（各列空值数量）】')
print(df.isnull().sum())

print('\n【3. 字段内容抽查（前 5 行）】')
print(df[['姓名', '职称', '教研室', '电子邮箱']].head(5).to_string())

print('\n【4. 异常值检查】')
bad_email = df[~df['电子邮箱'].str.contains('@', na=False) & df['电子邮箱'].ne('')]
print(f'  格式异常的邮箱：{len(bad_email)} 条')
print(f'  无邮箱记录：{df["电子邮箱"].eq("").sum()} 条')
print(f'  无教研室记录：{df["教研室"].eq("").sum()} 条（部分教师正常无此字段）')
print(f'  无职务记录：{df["职务"].eq("").sum()} 条（大多数教师正常无此字段）')

对照以下标准判断结果是否正常：

| 检查项 | 期望结果 | 若不符合，可能的原因 |
|--------|---------|--------------------|
| 数据量 | 几十至一百多条 | 若只有个位数，`find_all('div', class_='infors')` 的 class 名写错 |
| 格式异常邮箱 | 0 条 | 若有异常，`@` 识别逻辑出错，邮箱被填入其他列 |
| 姓名列内容 | 全为中文姓名 | 若出现邮箱或职称，字段严重错位 |
| 无教研室记录 | 0 条（所有教师都有教研室归属） | 若有空值，第一个 `p.one-line` 的解析逻辑出错 |
| 无职务记录 | 多数（大部分教师无行政职务） | 若为 0，可能把邮箱或教研室误判为职务 |

### 如果结果不对：发起调试对话{.unnumbered}

字段出错时，不要从头重写提示词，而是把具体错误现象告诉 AI。有效的调试消息包含三件事：

```
① 错误现象（哪列出了什么问题）
② 已做的验证（你确认了什么）
③ 中间结果（打印出的原始数据）
```

示例：

> 代码跑通了，但检查发现「教研室」列全部为空，共 0 条有值。我去页面确认，刘贯春的卡片里确实有「宏观经济教研室」。
>
> 我打印了刘贯春对应的 `one_lines` 文本列表：
> ```
> ['宏观经济教研室', '宏观经济教研室主任', 'liugch9@mail.sysu.edu.cn']
> ```
> 三个值都在，但教研室列还是空的。

这三件事让 AI 能精准定位问题，直接给出修复方案，而不是重写整段代码。信息越具体，修复越快。

---
## 课堂讨论

1. 本例用文本内容特征（含 `@`、含「教研室」）来识别字段，而不是靠 class 名或位置。这种方法有什么局限？如果某位教师的研究方向简介里也包含了「教研室」这个词，会发生什么？

2. 路径 A 和路径 B 最终得到的提示词完全一致。路径 B 花了六轮对话才到达这里，而路径 A 直接填表就完成了。对你来说，路径 B 的「额外成本」是否值得？在什么情况下路径 B 会更有价值？

3. 本例只请求了一次页面，没有设置请求间隔。如果改成爬取每位教师的**详情页**（每位教师点进去的个人页面），提示词里需要新增哪些内容？代码结构会有什么变化？